
# Round6 All-Class Recall — CLEAN

이 노트북은 기존 Round6에서 헷갈렸던 경로/압축해제/import 문제를 한 번에 정리한 버전입니다.

- 기존 계정에서 `/content/drive/MyDrive/02/ml/dataset`이 있으면 자동 사용
- 새 계정이면 `dataset.zip` 업로드 가능
- `round6_allclass_recall.zip`만 먼저 업로드하면 됨
- 클래스 순서 고정:
  `0 exit / 1 stair / 2 elevator / 3 extinguisher / 4 hydrant / 5 you_are_here / 6 door / 7 room`

**셀 1부터 순서대로 실행하세요.**


In [ ]:

# ============================================================
# CELL 1 — 설치 + Drive + Round6 패키지 준비
# ============================================================

!pip -q install ultralytics pyyaml opencv-python-headless

from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, sys, os, json, yaml

drive.mount("/content/drive")

PKG = Path("/content/round6_allclass_recall")

# 이미 풀려 있으면 그대로 사용
if not (PKG / "config.yaml").exists():
    zips = list(Path("/content").glob("round6_allclass_recall*.zip"))

    if not zips:
        print("📦 round6_allclass_recall.zip 을 업로드하세요.")
        uploaded = files.upload()
        zips = [
            Path("/content") / name
            for name in uploaded.keys()
            if name.lower().endswith(".zip")
            and "round6_allclass_recall" in name.lower()
        ]

    if not zips:
        raise RuntimeError("round6_allclass_recall.zip을 찾지 못했습니다.")

    zpath = max(zips, key=lambda p: p.stat().st_mtime)
    print("압축 해제:", zpath)

    # 패키지 zip은 /content에 바로 풀기
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall("/content")

assert (PKG / "config.yaml").exists(), "패키지 압축해제가 정상적으로 되지 않았습니다."
assert (PKG / "models" / "round2_best.pt").exists(), "Round2 모델이 없습니다."
assert (PKG / "models" / "round4_guarded_best.pt").exists(), "Round4 모델이 없습니다."

CONFIG = PKG / "config.yaml"

print()
print("✅ Round6 package 준비 완료")
print("PKG:", PKG)
print("CONFIG:", CONFIG)


In [ ]:

# ============================================================
# CELL 2 — Dataset 자동 탐색
# 기존 계정이면 Drive 원본을 자동 사용
# 새 계정이면 dataset.zip 업로드
# ============================================================

from pathlib import Path
from google.colab import files
import zipfile, shutil, yaml

IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

with open(CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def is_yolo_root(root):
    root = Path(root)
    return (
        (root / "images" / "train").exists()
        and (root / "labels" / "train").exists()
    )

def count_train(root):
    root = Path(root)
    ni = sum(
        1 for p in (root/"images"/"train").rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ) if (root/"images"/"train").exists() else 0
    nl = len(list((root/"labels"/"train").rglob("*.txt"))) if (root/"labels"/"train").exists() else 0
    return ni, nl

candidates = []

# 1) 예전 원본 경로 최우선
known = Path("/content/drive/MyDrive/02/ml/dataset")
if is_yolo_root(known):
    candidates.append(known)

# 2) config 경로
cfg_root = Path(cfg["paths"]["source_dataset_root"])
if is_yolo_root(cfg_root):
    candidates.append(cfg_root)

# 3) Drive + /content에서 탐색
for base in [Path("/content/drive/MyDrive"), Path("/content")]:
    if not base.exists():
        continue
    for p in base.rglob("images"):
        root = p.parent
        if is_yolo_root(root):
            candidates.append(root)

# 중복 제거
uniq = {}
for p in candidates:
    try:
        key = str(p.resolve())
    except:
        key = str(p)
    uniq[key] = p
candidates = list(uniq.values())

# 4) 없으면 dataset.zip 업로드
if not candidates:
    print("현재 계정에서 YOLO dataset을 찾지 못했습니다.")
    print("📦 기존 dataset 폴더를 압축한 dataset.zip을 업로드하세요.")
    uploaded = files.upload()

    dataset_zips = [
        Path("/content") / name
        for name in uploaded.keys()
        if name.lower().endswith(".zip")
        and "round6_allclass_recall" not in name.lower()
    ]

    if not dataset_zips:
        raise RuntimeError("dataset.zip이 업로드되지 않았습니다.")

    EXTRACT_ROOT = Path("/content/round6_dataset")
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(dataset_zips[0], "r") as z:
        z.extractall(EXTRACT_ROOT)

    for p in EXTRACT_ROOT.rglob("images"):
        root = p.parent
        if is_yolo_root(root):
            candidates.append(root)

if not candidates:
    raise RuntimeError(
        "YOLO dataset을 찾지 못했습니다. "
        "ZIP 안에 images/train 및 labels/train 구조가 있는지 확인하세요."
    )

# train 이미지가 가장 많은 dataset 선택
candidates = sorted(
    candidates,
    key=lambda p: count_train(p)[0],
    reverse=True
)
SOURCE_DATASET = candidates[0]

ni, nl = count_train(SOURCE_DATASET)

cfg["paths"]["source_dataset_root"] = str(SOURCE_DATASET)

# 새 계정에서도 결과가 사라지지 않도록 Drive에 저장
cfg["paths"]["output_root"] = "/content/drive/MyDrive/evacuation_yolo/round6_allclass_recall"

with open(CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print("======================================")
print("✅ DATASET READY")
print("======================================")
print("Dataset:", SOURCE_DATASET)
print("train images:", ni)
print("train labels:", nl)
print("Output:", cfg["paths"]["output_root"])

for split in ["train","val","test"]:
    idir = SOURCE_DATASET/"images"/split
    ldir = SOURCE_DATASET/"labels"/split
    nimg = sum(1 for p in idir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if idir.exists() else 0
    nlab = len(list(ldir.rglob("*.txt"))) if ldir.exists() else 0
    print(f"{split:5s}: images={nimg}, labels={nlab}")


In [ ]:

# ============================================================
# CELL 3 — Robust source-group locked split
# 같은 원본의 변형 이미지가 train/val/test에 섞이지 않도록 분리
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import random, shutil, re, json

CLASS_NAMES = [
    "exit","stair","elevator","extinguisher",
    "hydrant","you_are_here","door","room"
]
IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

SRC = Path(cfg["paths"]["source_dataset_root"])
OUT = Path(cfg["paths"]["output_root"])
LOCKED = OUT / "locked_dataset"
GROUP_RE = cfg.get("split",{}).get("group_regex", r"(evac_src\d+)")
SEED = int(cfg.get("split",{}).get("seed",3407))
VAL_RATIO = float(cfg.get("split",{}).get("val_ratio",0.15))
TEST_RATIO = float(cfg.get("split",{}).get("test_ratio",0.15))

def read_labels(path):
    rows=[]
    try:
        lines=Path(path).read_text(encoding="utf-8",errors="ignore").splitlines()
    except:
        return rows
    for line in lines:
        s=line.strip().split()
        if len(s)<5:
            continue
        try:
            cid=int(float(s[0]))
            vals=list(map(float,s[1:5]))
        except:
            continue
        if 0 <= cid < 8:
            rows.append((cid,*vals))
    return rows

def gid(stem):
    m=re.search(GROUP_RE,stem,flags=re.I)
    if m:
        return m.group(1).lower()
    s=stem.lower()
    s=re.sub(r"_v\d+.*$","",s)
    s=re.sub(r"_aug\d+.*$","",s)
    return s

pool=[]
for split in ["train","val","test"]:
    idir=SRC/"images"/split
    ldir=SRC/"labels"/split
    if not idir.exists() or not ldir.exists():
        continue

    labels_by_stem={p.stem:p for p in ldir.rglob("*.txt")}
    for img in idir.rglob("*"):
        if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS:
            continue
        lp=labels_by_stem.get(img.stem)
        if lp is None:
            continue
        labs=read_labels(lp)
        if not labs:
            continue
        pool.append({
            "image":img,
            "label":lp,
            "group":gid(img.stem),
            "labels":labs
        })

if not pool:
    raise RuntimeError("이미지-라벨 매칭이 0개입니다. dataset 구조를 확인해야 합니다.")

groups=defaultdict(list)
for x in pool:
    groups[x["group"]].append(x)

group_names=list(groups.keys())
rng=random.Random(SEED)

# 그룹별 포함 클래스
group_classes={}
for g,items in groups.items():
    present=set()
    for x in items:
        present.update(l[0] for l in x["labels"])
    group_classes[g]=present

n_groups=len(group_names)
n_val=max(1,round(n_groups*VAL_RATIO))
n_test=max(1,round(n_groups*TEST_RATIO))

def score_assignment(val_groups,test_groups):
    score=0
    for cid in range(8):
        vg=sum(cid in group_classes[g] for g in val_groups)
        tg=sum(cid in group_classes[g] for g in test_groups)
        if vg>0: score+=10
        if tg>0: score+=10
        score += min(vg,3) + min(tg,3)
    return score

# 여러 번 섞어 val/test class coverage가 가장 좋은 split 선택
best=None
best_score=-1
for _ in range(2000):
    names=group_names[:]
    rng.shuffle(names)
    val=names[:n_val]
    test=names[n_val:n_val+n_test]
    train=names[n_val+n_test:]
    sc=score_assignment(val,test)
    if sc>best_score:
        best_score=sc
        best=(train,val,test)

train_groups,val_groups,test_groups=best
assignment={"train":train_groups,"val":val_groups,"test":test_groups}

assert not (set(train_groups)&set(val_groups))
assert not (set(train_groups)&set(test_groups))
assert not (set(val_groups)&set(test_groups))

if LOCKED.exists():
    shutil.rmtree(LOCKED)

report={}
for split in ["train","val","test"]:
    oi=LOCKED/split/"images"
    ol=LOCKED/split/"labels"
    oi.mkdir(parents=True,exist_ok=True)
    ol.mkdir(parents=True,exist_ok=True)

    boxes=Counter()
    pos_imgs=Counter()
    n=0

    for g in assignment[split]:
        for idx,x in enumerate(groups[g]):
            new_name=f"{g}_{idx:03d}_{x['image'].name}"
            shutil.copy2(x["image"], oi/new_name)
            shutil.copy2(x["label"], ol/Path(new_name).with_suffix(".txt").name)

            present=set()
            for lab in x["labels"]:
                boxes[lab[0]]+=1
                present.add(lab[0])
            for cid in present:
                pos_imgs[cid]+=1
            n+=1

    report[split]={
        "groups":len(assignment[split]),
        "images":n,
        "box_counts":{CLASS_NAMES[c]:int(boxes[c]) for c in range(8)},
        "positive_images":{CLASS_NAMES[c]:int(pos_imgs[c]) for c in range(8)},
    }

OUT.mkdir(parents=True,exist_ok=True)
(OUT/"split_report.json").write_text(
    json.dumps(report,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

print("======================================")
print("✅ LOCKED SPLIT COMPLETE")
print("======================================")

for split in ["train","val","test"]:
    print(f"\n[{split.upper()}] groups={report[split]['groups']} images={report[split]['images']}")
    for cname,n in report[split]["box_counts"].items():
        print(f"{cname:15s}: {n}")

missing=[]
for split in ["val","test"]:
    for cname,n in report[split]["box_counts"].items():
        if n==0:
            missing.append(f"{split}:{cname}")

if missing:
    print("\n⚠️ 아래 클래스는 val/test GT가 없습니다.")
    print(", ".join(missing))
    print("해당 클래스 Recall은 이번 데이터만으로 신뢰성 있게 평가할 수 없습니다.")
else:
    print("\n✅ VAL/TEST 모두 8개 클래스 GT 존재")


In [ ]:

# ============================================================
# CELL 4 — Rare class balance
# fully-labeled train 이미지만 반복 복제
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import random, shutil, json

SRC_TRAIN = Path(cfg["paths"]["output_root"]) / "locked_dataset" / "train"
BAL = Path(cfg["paths"]["output_root"]) / "balanced_train"

if BAL.exists():
    shutil.rmtree(BAL)
shutil.copytree(SRC_TRAIN, BAL)

idir=SRC_TRAIN/"images"
ldir=SRC_TRAIN/"labels"
outi=BAL/"images"
outl=BAL/"labels"

TARGET = int(cfg.get("balance",{}).get("target_positive_images_per_class",250))
MAXREP = int(cfg.get("balance",{}).get("max_repeat_per_image",4))
rng=random.Random(int(cfg.get("split",{}).get("seed",3407)))

positives=defaultdict(list)

for img in idir.rglob("*"):
    if not img.is_file() or img.suffix.lower() not in IMAGE_EXTS:
        continue
    lp=ldir/img.with_suffix(".txt").name
    labs=read_labels(lp)
    present=set(x[0] for x in labs)
    for c in present:
        positives[c].append((img,lp))

added=Counter()

for c in range(8):
    if not positives[c]:
        continue
    need=max(0,TARGET-len(positives[c]))
    poolc=positives[c][:]
    rng.shuffle(poolc)
    repeat=Counter()
    k=0

    while added[c] < need and k < len(poolc)*MAXREP*3:
        img,lp=poolc[k % len(poolc)]
        key=str(img)
        if repeat[key] < MAXREP:
            name=f"bal_c{c}_{added[c]:04d}_{img.name}"
            shutil.copy2(img,outi/name)
            shutil.copy2(lp,outl/Path(name).with_suffix(".txt").name)
            repeat[key]+=1
            added[c]+=1
        k+=1

balance_report={
    "added_images_by_class":{CLASS_NAMES[c]:int(added[c]) for c in range(8)},
    "final_train_images":sum(1 for p in outi.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
}

(Path(cfg["paths"]["output_root"])/"balance_report.json").write_text(
    json.dumps(balance_report,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(balance_report,indent=2,ensure_ascii=False))
print("\n✅ Balance 완료")


In [ ]:

# ============================================================
# CELL 5 — Stage A / Stage B 학습
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch, yaml

OUT=Path(cfg["paths"]["output_root"])
RUNS=OUT/"runs"

device=0 if torch.cuda.is_available() else "cpu"
print("DEVICE:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

data_yaml=OUT/"round6_data.yaml"
data={
    "path":str(OUT),
    "train":"balanced_train/images",
    "val":"locked_dataset/val/images",
    "test":"locked_dataset/test/images",
    "names":{i:n for i,n in enumerate(CLASS_NAMES)}
}
data_yaml.write_text(
    yaml.safe_dump(data,sort_keys=False,allow_unicode=True),
    encoding="utf-8"
)

tc=cfg.get("train",{})
imgsz=int(tc.get("imgsz",896))
batch=int(tc.get("batch",2))
workers=int(tc.get("workers",1))

round2=PKG/"models"/"round2_best.pt"

print("\n===== STAGE A =====")
m=YOLO(str(round2))
m.train(
    data=str(data_yaml),
    imgsz=imgsz,
    batch=batch,
    workers=workers,
    cache=False,
    device=device,
    epochs=int(tc.get("stage_a_epochs",18)),
    freeze=int(tc.get("stage_a_freeze",10)),
    lr0=float(tc.get("stage_a_lr0",0.001)),
    project=str(RUNS),
    name="stage_a",
    exist_ok=True,
    amp=True,
    patience=8,
    hsv_h=.01,hsv_s=.18,hsv_v=.15,
    degrees=2,translate=.04,scale=.12,
    fliplr=.15,flipud=0,
    mosaic=.20,mixup=0,
    close_mosaic=4
)

stage_a=RUNS/"stage_a"/"weights"/"best.pt"
assert stage_a.exists(), "Stage A best.pt 생성 실패"

print("\n===== STAGE B =====")
m2=YOLO(str(stage_a))
m2.train(
    data=str(data_yaml),
    imgsz=imgsz,
    batch=batch,
    workers=workers,
    cache=False,
    device=device,
    epochs=int(tc.get("stage_b_epochs",30)),
    freeze=0,
    lr0=float(tc.get("stage_b_lr0",0.0002)),
    project=str(RUNS),
    name="stage_b",
    exist_ok=True,
    amp=True,
    patience=10,
    hsv_h=.005,hsv_s=.10,hsv_v=.10,
    degrees=1,translate=.03,scale=.08,
    fliplr=.10,flipud=0,
    mosaic=.10,mixup=0,
    close_mosaic=5
)

stage_b=RUNS/"stage_b"/"weights"/"best.pt"
assert stage_b.exists(), "Stage B best.pt 생성 실패"

print("\n✅ 학습 완료")
print("Stage A:",stage_a)
print("Stage B:",stage_b)


In [ ]:

# ============================================================
# CELL 6 — 8개 클래스별 최적 모델 + threshold 선택
# locked VAL로 선택 후 locked TEST 1회 평가
# ============================================================

from collections import defaultdict
from ultralytics import YOLO
import cv2, torch, json
from pathlib import Path

OUT=Path(cfg["paths"]["output_root"])
device=0 if torch.cuda.is_available() else "cpu"

def gt_boxes(label_path,W,H):
    out=defaultdict(list)
    for cid,xc,yc,bw,bh in read_labels(label_path):
        out[cid].append([
            (xc-bw/2)*W,(yc-bh/2)*H,
            (xc+bw/2)*W,(yc+bh/2)*H
        ])
    return out

def iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1])
    x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    bb=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    u=aa+bb-inter
    return inter/u if u>0 else 0.0

def greedy_match(preds,gts,iou_thr=.5):
    preds=sorted(preds,key=lambda x:x[4],reverse=True)
    used=set(); tp=fp=0
    for p in preds:
        best_j=-1; best_iou=0
        for j,g in enumerate(gts):
            if j in used:
                continue
            z=iou(p[:4],g)
            if z>best_iou:
                best_iou=z; best_j=j
        if best_j>=0 and best_iou>=iou_thr:
            used.add(best_j); tp+=1
        else:
            fp+=1
    fn=len(gts)-len(used)
    return tp,fp,fn

def collect_predictions(model_path,split):
    model=YOLO(str(model_path))
    rows=[]
    idir=OUT/"locked_dataset"/split/"images"
    ldir=OUT/"locked_dataset"/split/"labels"

    for img in sorted(idir.iterdir()):
        if img.suffix.lower() not in IMAGE_EXTS:
            continue
        im=cv2.imread(str(img))
        if im is None:
            continue
        H,W=im.shape[:2]
        gt=gt_boxes(ldir/img.with_suffix(".txt").name,W,H)

        res=model.predict(
            str(img),
            imgsz=int(cfg.get("train",{}).get("imgsz",896)),
            conf=.01,
            iou=.5,
            device=device,
            verbose=False
        )[0]

        pred=defaultdict(list)
        if res.boxes is not None and len(res.boxes):
            xy=res.boxes.xyxy.detach().cpu().numpy()
            cf=res.boxes.conf.detach().cpu().numpy()
            cl=res.boxes.cls.detach().cpu().numpy().astype(int)
            for b,c,k in zip(xy,cf,cl):
                pred[int(k)].append([
                    float(b[0]),float(b[1]),float(b[2]),float(b[3]),float(c)
                ])

        rows.append((pred,gt))
    return rows

def metric(rows,cid,thr):
    TP=FP=FN=0
    for pred,gt in rows:
        pp=[p for p in pred.get(cid,[]) if p[4]>=thr]
        tp,fp,fn=greedy_match(pp,gt.get(cid,[]),.5)
        TP+=tp; FP+=fp; FN+=fn

    precision=TP/(TP+FP) if TP+FP else 0.0
    recall=TP/(TP+FN) if TP+FN else 0.0
    f2=5*precision*recall/(4*precision+recall) if precision+recall else 0.0
    return {
        "threshold":thr,
        "precision":precision,
        "recall":recall,
        "f2":f2,
        "tp":TP,"fp":FP,"fn":FN
    }

thresholds=cfg.get("evaluation",{}).get(
    "thresholds",
    [0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50]
)
floors=cfg.get("evaluation",{}).get("precision_floor_by_class",{})

candidates=[
    ("round2",PKG/"models"/"round2_best.pt"),
    ("round4",PKG/"models"/"round4_guarded_best.pt"),
    ("stage_a",OUT/"runs"/"stage_a"/"weights"/"best.pt"),
    ("stage_b",OUT/"runs"/"stage_b"/"weights"/"best.pt"),
]
candidates=[x for x in candidates if x[1].exists()]

val_cache={}
val_best={}

print("VAL prediction collecting...")
for name,path in candidates:
    print(" -",name)
    rows=collect_predictions(path,"val")
    val_cache[name]=rows
    val_best[name]={}

    for cid,cname in enumerate(CLASS_NAMES):
        curve=[metric(rows,cid,float(t)) for t in thresholds]

        gt_total=max(m["tp"]+m["fn"] for m in curve) if curve else 0
        if gt_total==0:
            # GT 없는 클래스는 선택 대상에서 제외
            val_best[name][cname]=None
            continue

        floor=float(floors.get(cname,0.30))
        feasible=[m for m in curve if m["precision"]>=floor]

        if feasible:
            best=max(feasible,key=lambda m:(m["recall"],m["f2"],m["precision"],m["threshold"]))
        else:
            best=max(curve,key=lambda m:(m["f2"],m["recall"],m["precision"]))

        val_best[name][cname]=best

router={}
for cid,cname in enumerate(CLASS_NAMES):
    options=[]
    for name,path in candidates:
        m=val_best[name][cname]
        if m is not None:
            options.append((name,path,m))

    if not options:
        router[cname]={
            "class_id":cid,
            "model_name":"UNAVAILABLE",
            "model_path":"",
            "threshold":None,
            "val_metrics":None
        }
        continue

    name,path,best=max(
        options,
        key=lambda x:(x[2]["recall"],x[2]["f2"],x[2]["precision"],x[2]["threshold"])
    )

    router[cname]={
        "class_id":cid,
        "model_name":name,
        "model_path":str(path),
        "threshold":best["threshold"],
        "val_metrics":best
    }

print("\nTEST prediction collecting...")
test_cache={}
for name,path in candidates:
    print(" -",name)
    test_cache[name]=collect_predictions(path,"test")

test_metrics={}
for cid,cname in enumerate(CLASS_NAMES):
    r=router[cname]

    if r["model_name"]=="UNAVAILABLE":
        test_metrics[cname]={
            "threshold":None,
            "precision":None,
            "recall":None,
            "f2":None,
            "tp":0,"fp":0,"fn":0,
            "note":"No GT in validation"
        }
        continue

    test_metrics[cname]=metric(
        test_cache[r["model_name"]],
        cid,
        r["threshold"]
    )

report={
    "router":router,
    "test_metrics":test_metrics,
    "class_order":CLASS_NAMES
}

valid_recalls=[
    m["recall"]
    for m in test_metrics.values()
    if m.get("recall") is not None
]

report["all_evaluable_classes_recall_1_0"] = (
    len(valid_recalls)>0
    and all(abs(x-1.0)<1e-12 for x in valid_recalls)
)

(OUT/"class_router.json").write_text(
    json.dumps(router,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

(OUT/"allclass_selection_report.json").write_text(
    json.dumps(report,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

print("\n======================================")
print("FINAL TEST RESULT")
print("======================================")

for cname,m in test_metrics.items():
    if m["recall"] is None:
        print(f"{cname:15s}: 평가불가 (VAL GT 없음)")
    else:
        print(
            f"{cname:15s} "
            f"Recall={m['recall']:.4f} "
            f"Precision={m['precision']:.4f} "
            f"F2={m['f2']:.4f} "
            f"TP={m['tp']} FP={m['fp']} FN={m['fn']}"
        )

print("\nALL EVALUABLE RECALL == 1.0:",
      report["all_evaluable_classes_recall_1_0"])

print("\n✅ 저장:")
print(OUT/"class_router.json")
print(OUT/"allclass_selection_report.json")


In [ ]:

# ============================================================
# CELL 7 — 최종 클래스별 사용 모델 확인
# ============================================================

import json
from pathlib import Path

OUT=Path(cfg["paths"]["output_root"])
router=json.loads((OUT/"class_router.json").read_text(encoding="utf-8"))
report=json.loads((OUT/"allclass_selection_report.json").read_text(encoding="utf-8"))

print("======================================")
print("CLASS ROUTER")
print("======================================")

for cname,r in router.items():
    if r["model_name"]=="UNAVAILABLE":
        print(f"{r['class_id']} {cname:15s} -> 평가불가")
    else:
        print(
            f"{r['class_id']} {cname:15s} -> "
            f"{r['model_name']:8s} "
            f"conf={r['threshold']:.2f} "
            f"VAL Recall={r['val_metrics']['recall']:.4f}"
        )

print("\n======================================")
print("TEST RECALL")
print("======================================")

for cname,m in report["test_metrics"].items():
    if m["recall"] is None:
        print(f"{cname:15s}: 평가불가")
    else:
        print(f"{cname:15s}: {m['recall']:.4f}")

print("\n결과 폴더:")
print(OUT)


In [ ]:
# ============================================================
# Round6 -> VS Code 배포용 모델 패키지 생성
# ============================================================

from pathlib import Path
import shutil
import zipfile
import hashlib
import json
from google.colab import files

ROUND6 = Path(
    "/content/drive/MyDrive/evacuation_yolo/round6_allclass_recall"
)

PKG = Path("/content/round6_allclass_recall")

EXPORT = Path("/content/round6_models_for_vscode")

if EXPORT.exists():
    shutil.rmtree(EXPORT)

EXPORT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 필요한 파일
# ------------------------------------------------------------

sources = {
    "round2_best.pt":
        PKG / "models" / "round2_best.pt",

    "stage_a_best.pt":
        ROUND6 / "runs" / "stage_a" / "weights" / "best.pt",

    "stage_b_best.pt":
        ROUND6 / "runs" / "stage_b" / "weights" / "best.pt",

    "class_router.json":
        ROUND6 / "class_router.json",

    "allclass_selection_report.json":
        ROUND6 / "allclass_selection_report.json",
}


print("======================================")
print("ROUND6 EXPORT CHECK")
print("======================================")

missing = []

for name, src in sources.items():

    if src.exists():
        print("✅", name)
        print("   ", src)
    else:
        print("❌", name)
        print("   ", src)
        missing.append(name)


if missing:
    raise RuntimeError(
        "\n필수 파일이 없습니다: "
        + ", ".join(missing)
    )


# ------------------------------------------------------------
# 파일 복사
# ------------------------------------------------------------

for name, src in sources.items():
    shutil.copy2(
        src,
        EXPORT / name
    )


# ------------------------------------------------------------
# SHA-256 생성
# ------------------------------------------------------------

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


hashes = {}

for name in [
    "round2_best.pt",
    "stage_a_best.pt",
    "stage_b_best.pt",
]:

    hashes[name] = sha256(
        EXPORT / name
    )


(EXPORT / "model_hashes.json").write_text(
    json.dumps(
        hashes,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Router / TEST 결과 콘솔 출력
# ------------------------------------------------------------

router = json.loads(
    (EXPORT / "class_router.json").read_text(
        encoding="utf-8"
    )
)

report = json.loads(
    (
        EXPORT /
        "allclass_selection_report.json"
    ).read_text(
        encoding="utf-8"
    )
)


print()
print("======================================")
print("CLASS ROUTER")
print("======================================")

for cname, r in router.items():

    print(
        f"{r['class_id']} "
        f"{cname:15s} -> "
        f"{r['model_name']:8s} "
        f"conf={r['threshold']}"
    )


print()
print("======================================")
print("TEST METRICS")
print("======================================")

for cname, m in report["test_metrics"].items():

    print(
        f"{cname:15s} "
        f"P={m['precision']:.4f} "
        f"R={m['recall']:.4f} "
        f"F2={m['f2']:.4f} "
        f"TP={m['tp']} "
        f"FP={m['fp']} "
        f"FN={m['fn']}"
    )


print()
print("======================================")
print("MODEL SHA-256")
print("======================================")

for name, h in hashes.items():
    print(name)
    print(h)


# ------------------------------------------------------------
# README
# ------------------------------------------------------------

readme = """
Round6 VS Code Deployment Models

Files:
- round2_best.pt
- stage_a_best.pt
- stage_b_best.pt
- class_router.json
- allclass_selection_report.json
- model_hashes.json

Class order MUST remain:

0 exit
1 stair
2 elevator
3 extinguisher
4 hydrant
5 you_are_here
6 door
7 room

Do not rename class IDs or reorder classes.
"""

(EXPORT / "README.txt").write_text(
    readme.strip(),
    encoding="utf-8"
)


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

ZIP_PATH = Path(
    "/content/round6_models_for_vscode.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for p in EXPORT.rglob("*"):

        if p.is_file():

            z.write(
                p,
                p.relative_to(EXPORT)
            )


print()
print("======================================")
print("✅ EXPORT COMPLETE")
print("======================================")

print(ZIP_PATH)

print(
    "ZIP size:",
    round(
        ZIP_PATH.stat().st_size
        / 1024 / 1024,
        2
    ),
    "MB"
)

print()
print("⬇️ 다운로드 시작")


files.download(
    str(ZIP_PATH)
)

In [ ]:
# ============================================================
# Round7 inference용 Round6 모델 자동 복구
# ============================================================

from google.colab import files
from pathlib import Path
import zipfile
import shutil

PKG = Path("/content/round6_allclass_recall")

round2_path = PKG / "models" / "round2_best.pt"

if not round2_path.exists():

    print("📦 round6_allclass_recall.zip 을 업로드하세요.")

    uploaded = files.upload()

    zips = [
        Path("/content") / name
        for name in uploaded.keys()
        if name.lower().endswith(".zip")
        and "round6_allclass_recall" in name.lower()
    ]

    if not zips:
        raise RuntimeError(
            "❌ round6_allclass_recall.zip을 찾지 못했습니다."
        )

    ZIP_PATH = zips[0]

    print("압축 해제:", ZIP_PATH)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/content")


print()
print("======================================")
print("MODEL CHECK")
print("======================================")

paths = {
    "round2":
        Path(
            "/content/round6_allclass_recall/"
            "models/round2_best.pt"
        ),

    "stage_a":
        Path(
            "/content/drive/MyDrive/"
            "evacuation_yolo/"
            "round6_allclass_recall/"
            "runs/stage_a/weights/best.pt"
        ),

    "stage_b":
        Path(
            "/content/drive/MyDrive/"
            "evacuation_yolo/"
            "round6_allclass_recall/"
            "runs/stage_b/weights/best.pt"
        ),

    "router":
        Path(
            "/content/drive/MyDrive/"
            "evacuation_yolo/"
            "round6_allclass_recall/"
            "class_router.json"
        ),
}

for name, p in paths.items():

    print(
        "✅" if p.exists() else "❌",
        name,
        p
    )


missing = [
    name
    for name, p in paths.items()
    if not p.exists()
]


if missing:
    raise RuntimeError(
        "❌ 아직 없는 파일: "
        + ", ".join(missing)
    )


print()
print("✅ Round7 inference에 필요한 모델 준비 완료")
print("➡️ 이제 Round7 마지막 셀을 다시 실행하세요.")

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

targets = [
    "class_router.json",
    "allclass_selection_report.json",
    "best.pt",
]

print("======================================")
print("ROUND6 FILE SEARCH")
print("======================================")

found = []

for p in ROOT.rglob("*"):
    if not p.is_file():
        continue

    s = str(p).lower()

    if (
        p.name in ["class_router.json", "allclass_selection_report.json"]
        or (
            p.name == "best.pt"
            and (
                "stage_a" in s
                or "stage_b" in s
                or "round6" in s
            )
        )
    ):
        found.append(p)

for p in found:
    print(p)

print()
print("발견된 파일 수:", len(found))

In [ ]:
from pathlib import Path
import shutil
import zipfile
from google.colab import files

ROUND6 = Path(
    "/content/drive/MyDrive/evacuation_yolo/round6_allclass_recall"
)

EXPORT = Path("/content/round6_runtime_files")
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)

srcs = {
    "stage_a_best.pt": ROUND6/"runs/stage_a/weights/best.pt",
    "stage_b_best.pt": ROUND6/"runs/stage_b/weights/best.pt",
    "class_router.json": ROUND6/"class_router.json",
    "allclass_selection_report.json": ROUND6/"allclass_selection_report.json",
}

for name, src in srcs.items():
    if not src.exists():
        raise RuntimeError(f"없음: {src}")
    shutil.copy2(src, EXPORT/name)
    print("✅", name)

ZIP = Path("/content/round6_runtime_files.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in EXPORT.iterdir():
        z.write(p, p.name)

print("✅ 생성:", ZIP)
files.download(str(ZIP))